In [3]:
import pandas as pd
import os
import openpyxl
import pygwalker as pyg
import numpy as np
import re
import sys
from pathlib import Path

In [4]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn

In [10]:
# Vegetation zonal stats output
veg_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run__20260224/'
veg_parquet_name = f'veg_model_zonal_stats_v{cn.veg_model_version_underscore}_20260224_19_21_07.parquet'

In [5]:
# AGC emission and removal factor zonal stats output
veg_EF_RF_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__EF_RF_only__20260506/'
EF_RF_parquet_name = f'veg_model_zonal_stats_v{cn.veg_model_version_underscore}_20260506_19_58_38.parquet'

In [11]:
# Reads vegetation zonal stats parquet table
veg_df_raw = pd.read_parquet(f'{veg_zonal_stats_folder}{veg_parquet_name}')

# Because some contextual rows are blank
# Shouldn't be necessary if I run vegetation zonal stats again and create new parquet files 
veg_df_raw["continent_ecozone"] = veg_df_raw["continent_ecozone"].fillna("Unassigned")  
veg_df_raw["country_name"] = veg_df_raw["country_name"].fillna("Unassigned")  
veg_df_raw['region'] = veg_df_raw['region'].fillna("Unassigned")

In [6]:
# Reads EF and RF zonal stats parquet table
EF_RF_df_raw = pd.read_parquet(f'{veg_EF_RF_zonal_stats_folder}{EF_RF_parquet_name}')

# # Because some contextual rows are blank
# # Shouldn't be necessary if I run vegetation zonal stats again and create new parquet files 
# EF_RF_df_raw["continent_ecozone"] = EF_RF_df_raw["continent_ecozone"].fillna("Unassigned")  
# EF_RF_df_raw["country_name"] = EF_RF_df_raw["country_name"].fillna("Unassigned")  
# EF_RF_df_raw['region'] = EF_RF_df_raw['region'].fillna("Unassigned")

In [15]:
EF_RF_df_raw.columns

Index(['analysis_layer', 'adm0', 'land_state_node', 'WDPA', 'cont_eco',
       'Landmark', 'starting_composite_primary_forest', 'year', 'value',
       'tile_id', 'area_ha', 'land_state_meaning', 'land_state_broad_class',
       'land_state_detailed_class', 'tall_veg_type', 'gas', 'country_name',
       'region', 'continent', 'continent_ecozone', 'climate_domain',
       'WDPA_type', 'WDPA_high_protection', 'density__Mg_ha'],
      dtype='object')

In [14]:
### Areas of forest change, annual carbon carbon densities, and annual fluxes by land state and year for uncertainty analysis (main data source)

veg_df_uncert = veg_df_raw.copy()

veg_by_land_state_year = (
     veg_df_uncert   
    .groupby(["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "year", "analysis_layer"], as_index=False, dropna=False)
    .agg({"area_ha": "sum", "value": "sum"})
)
veg_by_land_state_year["area_Mha"] = veg_by_land_state_year["area_ha"] / 1e6

veg_by_land_state_year.to_csv("/mnt/c/GIS/veg_outputs_by_land_state_year_for_uncert_analysis.csv", index=False)  # Export area as csv 
veg_by_land_state_year

,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,year,analysis_layer,area_ha,value,area_Mha
0,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,carbon_density__non_soil__MgC_ha,1.620982e+05,1.136252e+06,0.162098
1,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__AGC__MgCO2,1.620982e+05,-2.413498e+06,0.162098
2,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__BGC__MgCO2,1.620982e+05,-1.248027e+06,0.162098
3,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__all_C_pools__MgCO2,1.620982e+05,-4.031480e+06,0.162098
4,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,gross_removals__deadwood_C__MgCO2,1.620982e+05,-3.473215e+05,0.162098
...,...,...,...,...,...,...,...,...,...
7228,70000000,Not in decision tree,no_flux,no_flux,2020,carbon_density__non_soil__MgC_ha,2.357357e+09,1.713759e+06,2357.356445
7229,70000000,Not in decision tree,no_flux,no_flux,2021,carbon_density__non_soil__MgC_ha,2.465766e+09,2.295650e+06,2465.766113
7230,70000000,Not in decision tree,no_flux,no_flux,2022,carbon_density__non_soil__MgC_ha,2.463946e+09,2.464142e+06,2463.946533
7231,70000000,Not in decision tree,no_flux,no_flux,2023,carbon_density__non_soil__MgC_ha,2.450529e+09,2.736400e+06,2450.528564


In [23]:
### Areas of forest change, annual carbon carbon densities, and annual fluxes by land state and continent-ecozone for uncertainty analysis (partial disturbance emission factor uncertainty)

veg_df_uncert = veg_df_raw.copy()

veg_by_land_state_conteco = (
     veg_df_uncert   
    .groupby(["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "cont_eco", "continent_ecozone", "analysis_layer"], as_index=False, dropna=False)
    .agg({"area_ha": "sum", "value": "sum"})
)

veg_by_land_state_conteco = veg_by_land_state_conteco[veg_by_land_state_conteco["analysis_layer"] == "gross_emissions__AGC__MgCO2"].reset_index(drop=True)
veg_by_land_state_conteco = veg_by_land_state_conteco[veg_by_land_state_conteco["land_state_detailed_class"] == "tree_tree_disturbed"].reset_index(drop=True)

veg_by_land_state_conteco.to_csv("/mnt/c/GIS/veg_outputs_by_land_state_conteco_for_uncert_analysis.csv", index=False)  # Export area as csv 
veg_by_land_state_conteco

,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,cont_eco,continent_ecozone,analysis_layer,area_ha,value
0,42112120,Oil palm partially disturbed in the current in...,tree,tree_tree_disturbed,0,unassigned,gross_emissions__AGC__MgCO2,58.442814,1.263068e+04
1,42112120,Oil palm partially disturbed in the current in...,tree,tree_tree_disturbed,1017,Tropical dry forest,gross_emissions__AGC__MgCO2,300.935181,2.201669e+04
2,42112120,Oil palm partially disturbed in the current in...,tree,tree_tree_disturbed,1018,Tropical moist deciduous forest,gross_emissions__AGC__MgCO2,158.002319,9.054861e+03
3,42112120,Oil palm partially disturbed in the current in...,tree,tree_tree_disturbed,1019,Tropical mountain system,gross_emissions__AGC__MgCO2,408.000336,7.293634e+04
4,42112120,Oil palm partially disturbed in the current in...,tree,tree_tree_disturbed,1020,Tropical rainforest,gross_emissions__AGC__MgCO2,61300.730469,4.867904e+06
...,...,...,...,...,...,...,...,...,...
434,42122290,Trees outside forests partially disturbed in t...,tree,tree_tree_disturbed,7011,Temperate continental forest,gross_emissions__AGC__MgCO2,117.942642,7.560568e+03
435,42122290,Trees outside forests partially disturbed in t...,tree,tree_tree_disturbed,7013,Temperate mountain system,gross_emissions__AGC__MgCO2,71.012375,8.360490e+03
436,42122290,Trees outside forests partially disturbed in t...,tree,tree_tree_disturbed,7014,Temperate oceanic forest,gross_emissions__AGC__MgCO2,814.111755,5.994646e+04
437,42122290,Trees outside forests partially disturbed in t...,tree,tree_tree_disturbed,7015,Temperate steppe,gross_emissions__AGC__MgCO2,56.776386,1.445159e+03


In [16]:
### EF and RF by land state and year for uncertainty analysis

EF_RF_df_uncert = EF_RF_df_raw.copy()

EF_RF_by_land_state_year = (
     EF_RF_df_uncert   
    .groupby(["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "year", "analysis_layer"], as_index=False, dropna=False)
    .agg({"area_ha": "sum", "value": "sum"})
)
EF_RF_by_land_state_year["EF_RF"] = EF_RF_by_land_state_year["value"]/EF_RF_by_land_state_year["area_ha"]

EF_RF_by_land_state_year.to_csv("/mnt/c/GIS/EF_RF_outputs_by_land_state_year_for_uncert_analysis.csv", index=False)  # Export area as csv 
EF_RF_by_land_state_year

,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,year,analysis_layer,area_ha,value,EF_RF
0,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2016,removal_factor__AGC__MgC,162098.156250,658226.562500,4.060667
1,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2017,removal_factor__AGC__MgC,138574.140625,557368.812500,4.022171
2,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2018,removal_factor__AGC__MgC,162225.312500,666010.937500,4.105469
3,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2019,removal_factor__AGC__MgC,182011.734375,750203.437500,4.121731
4,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,2020,removal_factor__AGC__MgC,191058.765625,795546.750000,4.163885
...,...,...,...,...,...,...,...,...,...
921,62290000,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,2020,AGC_emission_factor_CO2_only__fraction,48784.687500,48784.687500,1.000000
922,62290000,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,2021,AGC_emission_factor_CO2_only__fraction,20965.435547,20965.435547,1.000000
923,62290000,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,2022,AGC_emission_factor_CO2_only__fraction,14441.843750,14441.843750,1.000000
924,62290000,Short vegetation loss converted to non-water w...,short_veg,short_veg_loss,2023,AGC_emission_factor_CO2_only__fraction,17373.843750,17373.843750,1.000000


In [18]:
### Area by country and land state and year for removal factors for uncertainty analysis

iso_land_state_area_df_uncert = veg_df_raw.copy()

iso_land_state_area = (
     iso_land_state_area_df_uncert   
    .groupby(["adm0", "land_state_detailed_class", "analysis_layer"], as_index=False, dropna=False)
    .agg({"area_ha": "sum", "value": "sum"})
)

iso_land_state_area.to_csv("/mnt/c/GIS/iso_land_state_area_for_uncert_analysis.csv", index=False)  # Export area as csv 
iso_land_state_area

,adm0,land_state_detailed_class,analysis_layer,area_ha,value
0,ABW,short_veg_gain,carbon_density__non_soil__MgC_ha,3.607079e+00,1.275795e+01
1,ABW,short_veg_gain,gross_removals__AGC__MgCO2,3.607079e+00,-1.236690e+01
2,ABW,short_veg_gain,gross_removals__BGC__MgCO2,3.607079e+00,-3.441226e+01
3,ABW,short_veg_gain,gross_removals__all_C_pools__MgCO2,3.607079e+00,-4.677916e+01
4,ABW,short_veg_gain,net_flux__AGC__MgCO2,3.607079e+00,-1.236690e+01
...,...,...,...,...,...
24565,ZWE,tree_tree_undisturbed,net_flux__BGC__MgCO2,1.680803e+08,-1.176224e+08
24566,ZWE,tree_tree_undisturbed,net_flux__all_C_pools__CO2_only__MgCO2,1.765199e+08,-3.705679e+08
24567,ZWE,tree_tree_undisturbed,net_flux__all_C_pools__all_gases__MgCO2e,1.765199e+08,-3.654841e+08
24568,ZWE,tree_tree_undisturbed,net_flux__deadwood_C__MgCO2,9.741758e+05,-2.945335e+04
